<a href="https://colab.research.google.com/github/dewmini-06/Statistical-Learning-e22317/blob/main/Assignment7d_E22317.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 3 – Bayesian Estimation for Structural Health Monitoring via Bounded Grid Updates

---

## 1. Prior Belief Boundaries

The prior distribution for the remaining stiffness efficiency is

$$
\Theta \sim \mathrm{Beta}(8,1.5)
$$

with probability density function

$$
f(\theta)=
\frac{1}{B(8,1.5)}
\theta^{7}(1-\theta)^{0.5},
\qquad
0<\theta<1.
$$

The prior mean is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
=
0.842.
$$

### Interpretation

The prior mean of **0.842** indicates that the structure is expected to retain approximately **84.2%** of its original stiffness before any measurements are collected. Since the Beta(8,1.5) distribution places most of its probability near 1, it appropriately reflects the engineering assumption that a newly installed or well-maintained component is likely to be in a healthy condition while still allowing for some uncertainty.

---

## 2. Structural Likelihood Formulation

The measurement model is

$$
y_k=\theta K_{\text{nominal}}e^{\varepsilon_k},
$$

where

$$
\varepsilon_k\sim N(0,\sigma^2).
$$

Taking the natural logarithm,

$$
\ln(y_k)
=
\ln(\theta K_{\text{nominal}})
+
\varepsilon_k.
$$

The likelihood of a single observation is

$$
L(y_k|\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\ln y_k-\ln(\theta K_{\text{nominal}})\right)^2}
{2\sigma^2}
\right].
$$

The joint likelihood for all observations is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
L(y_i|\theta).
$$

---

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

Since the prior is **Beta** while the likelihood is **Log-Normal**, they are **not conjugate**. Multiplying these distributions does not produce another Beta distribution, so there is no closed-form analytical posterior.

Using Bayes' theorem,

$$
f(\theta|y^{(k)})
\propto
L(y_k|\theta)
f(\theta|y^{(k-1)}).
$$

Expanding,

$$
f(\theta|y^{(k)})
\propto
\left[
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left(
-\frac{\left(\ln y_k-\ln(\theta K_{\text{nominal}})\right)^2}
{2\sigma^2}
\right)
\right]
f(\theta|y^{(k-1)}).
$$

Hence, numerical methods are required to evaluate the posterior distribution.

---

## 4. Running Point Estimates

### Posterior Mean

The Bayesian posterior mean is

$$
\boxed{
\hat{\theta}_{\mathrm{Bayes}}
=
\int_{0}^{1}
\theta\,
f(\theta|y^{(k)})
\,d\theta
}
$$

### Maximum A Posteriori (MAP)

The MAP estimate is

$$
\boxed{
\hat{\theta}_{\mathrm{MAP}}
=
\operatorname*{arg\,max}_{0<\theta<1}
f(\theta|y^{(k)})
}
$$

Since no analytical solution exists, both estimates are computed numerically from the posterior grid.

---

## 5. Algorithmic Grid Approximation and Normalization

The posterior distribution can be approximated using the following numerical procedure:

1. Create a fixed grid of values over the interval **0.01 ≤ θ ≤ 1.0**.
2. Evaluate the Beta prior at every grid point.
3. For each new sensor measurement:
   - Compute the log-normal likelihood across the grid.
   - Multiply the previous posterior by the likelihood to obtain the unnormalized posterior.
4. Normalize the posterior using the trapezoidal rule:

$$
f_{\text{new}}(\theta)
=
\frac{\tilde{f}(\theta)}
{\int_{0.01}^{1.0}\tilde{f}(\theta)\,d\theta}.
$$

where the integral is computed numerically using **`np.trapezoid()`**.

5. Compute the posterior mean using numerical integration.
6. Obtain the MAP estimate by selecting the grid point with the highest posterior probability.
7. Repeat the process after each new sensor measurement.

The lower boundary of **0.01** prevents numerical instability caused by evaluating $\ln(0)$, while the upper boundary of **1.0** reflects the physical limit that the stiffness efficiency cannot exceed its original value.

---

## 6. Performance Tracking and Degradation Convergence Analysis

###



As additional sensor measurements are collected, the posterior distribution gradually shifts from the optimistic prior toward the true stiffness value of **0.68**. Initially, the prior strongly favors healthy structural conditions, but repeated measurements provide increasing evidence of degradation. After several observations, both the Posterior Mean and MAP estimate converge close to **0.68**, while the posterior distribution becomes noticeably narrower.

The narrowing of the posterior indicates a reduction in uncertainty and increasing confidence in the estimated structural condition. Once the estimates stabilize near **0.68**, the monitoring system has accumulated sufficient evidence to conclude that the component has experienced significant stiffness degradation, enabling engineers to make informed maintenance or safety decisions.